In [1]:
import pickle
import pandas as pd
import numpy as np
from pypcd4 import PointCloud
import cv2
from PIL import Image
import json
import math

In [2]:
# Необходимые столбцы
CUBOIDS_COLUMNS = ['label', 'yaw', 'position.x', 'position.y', 'dimensions.x', 'dimensions.y']
pcd_columns = ['x', 'y', 'z', 'i']
BEV_LABELS = ['Car', 'Bus', 'Pedestrian', 'Bus', 'Pickup Truck']

# Квадрат для препроцессинга
PCD_AREA_WIDTH = 50
PCD_AREA_HEIGHT = 50

# Размер изображения после препроцессинга
BEV_WIDTH = 640
BEV_HEIGHT = 640

# Ограничения по дальности лидара
WIDTH_MAGIC = 50
HEIGHT_MAGIC = 50

In [236]:
cur_folder = '002'
cur_file = '00.pkl'

with open(f'./training/Pandaset/{cur_folder}/annotations/cuboids/{cur_file}', 'rb') as f:

    cuboids_data = pickle.load(f)
    # print(set(cuboids_data['cuboids.sibling_id']))
    # print(cuboids_data.columns)
    # cuboids_data = cuboids_data[cuboids_data['cuboids.sibling_id'].isin(['6c71e7b5-8eb3-4d3c-876f-e258e0e399db', '52195828-398a-4f69-a0ee-7ba9ce260ee1', '64b6e696-e63f-406c-b5f3-a8ca3668de3d', 'e5ad8a8c-8a62-4e7e-a08b-3a8a51ba0fd6',])]
    cuboids_data = cuboids_data[cuboids_data['cuboids.sensor_id'].isin([-1, 0])]
    cuboids_data = cuboids_data[CUBOIDS_COLUMNS]
    cuboids_data = cuboids_data[cuboids_data['label'].isin(['Car', 'Pedestrian', 'Bus', 'Pickup Truck'])]
    cuboids_array = cuboids_data.to_numpy()

with open(f'./training/Pandaset/{cur_folder}/lidar/{cur_file}', 'rb') as f:
    lidar_data = pickle.load(f)
    points_array = lidar_data[pcd_columns].to_numpy()
    
with open(f"./training/Pandaset/{cur_folder}/lidar/poses.json", 'r') as f:
    pose_data = json.load(f)
    pose_data = pose_data[int(cur_file[:2])]
    
    # 'd193e4e5-6794-4915-8772-cd64fe175259', '90f0d4a4-7a07-47c2-8636-3b79c468aea1', '0c23bb23-696c-4e11-bca3-184b877ca0d9', '0c045e66-3cd6-4dbb-84eb-fc3eb64d6ef1', '4ca78c65-b17e-4cc4-95d8-556e9dea834a', '6c1478ec-63e4-467e-aea9-67c4e8ebc5a3', 'f12d87c9-9522-4316-a660-7a5ec80fd8da', '5bc135e3-0e44-4e78-8a67-cc06b97cb180', '6c71e7b5-8eb3-4d3c-876f-e258e0e399db', '52195828-398a-4f69-a0ee-7ba9ce260ee1', '64b6e696-e63f-406c-b5f3-a8ca3668de3d', 'e5ad8a8c-8a62-4e7e-a08b-3a8a51ba0fd6', '3fe53d03-50b8-42eb-964e-d8f779df3a0f', 'f2b27077-6e42-42da-b044-6fa1fe100790', '580459ea-a1cc-460f-8a19-c5b645e09b5b', '7d4570ba-1ef6-4d19-81bf-de716311e966', '-', 'e7b62fa0-53fb-4ad8-93b8-971fdda41426', 'b2aa81b7-2c05-4e5e-a162-231b4183ae35', 'cbb288dd-af23-411d-b7ea-08c461c0052c', '3d8dc08d-36e3-4d29-8998-c54f938209b6', 'fe0752c6-def9-4cd5-98a0-cccb49476e41', 'c60b3e91-ce10-499e-889e-1a811a8cf489', '8a9f91ec-f724-447e-826c-b062a0fb8a4d', '316a2d28-6309-444a-b596-6c7df94a1d3f', '64aaf11d-2a76-4deb-bf41-320e92633a1a'

/tmp/ipykernel_105466/2342678219.py:6: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  cuboids_data = pickle.load(f)
/tmp/ipykernel_105466/2342678219.py:16: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  lidar_data = pickle.load(f)


In [237]:
# def poses_to_rads(poses_data):
#     rads = []
#     for pose_data in poses_data:
#         yaw = get_z_rotation(pose_data['heading'])
#         rads.append(yaw)
    
#     return np.asarray(rads)

    

def get_z_rotation(heading):
    w = heading['w']
    x = heading['x']
    y = heading['y']
    z = heading['z']
    vx = 1 - 2 * (y**2 + z**2)
    vy = 2 * (x * y + w * z)
    angle_rad = math.atan2(vy, vx)
    # angle_deg = math.degrees(angle_rad)
    
    return angle_rad

yaw_bias = get_z_rotation(pose_data['heading'])

In [238]:
def xywhr_to_4xy(xywhr):
    def rotate_x(a):
        cos_a = np.cos(a)
        sin_a = np.sin(a)
        return lambda x, y: x * cos_a + y * sin_a

    def rotate_y(a):
        cos_a = np.cos(a)
        sin_a = np.sin(a)
        return lambda x, y: -x * sin_a + y * cos_a

    start_x1 = - xywhr[:, 2] / 2
    start_x2 = xywhr[:, 2] / 2
    start_y1 = xywhr[:, 3] / 2
    start_y2 = -xywhr[:, 3] / 2
    
    yaw = xywhr[:, 4]
    
    # new_a = yaw - yaw_bias
    new_a = yaw - np.pi / 2
    rotate_x_func = rotate_x(new_a)
    rotate_y_func = rotate_y(new_a)
    
    points_4xy = np.zeros((xywhr.shape[0], 8))
    
    points_4xy[:, 0] = np.round(rotate_x_func(start_x1, start_y1), 4) + xywhr[:, 1]
    points_4xy[:, 2] = np.round(rotate_x_func(start_x2, start_y1), 4) + xywhr[:, 1]
    points_4xy[:, 4] = np.round(rotate_x_func(start_x1, start_y2), 4) + xywhr[:, 1]
    points_4xy[:, 6] = np.round(rotate_x_func(start_x2, start_y2), 4) + xywhr[:, 1]
    
    points_4xy[:, 1] = np.round(rotate_y_func(start_x1, start_y1), 4) + xywhr[:, 0]
    points_4xy[:, 3] = np.round(rotate_y_func(start_x2, start_y1), 4) + xywhr[:, 0]
    points_4xy[:, 5] = np.round(rotate_y_func(start_x1, start_y2), 4) + xywhr[:, 0]
    points_4xy[:, 7] = np.round(rotate_y_func(start_x2, start_y2), 4) + xywhr[:, 0]

    return points_4xy


def get_label_indexes(label_col):
    label_indexes = np.asarray([BEV_LABELS.index(x) for x in label_col])
    return label_indexes

def get_normalized_cuboid(cuboid_data, row, col, x_min_border, y_min_border):
    x_min_area_border = x_min_border + row * PCD_AREA_WIDTH
    y_min_area_border = y_min_border + col * PCD_AREA_HEIGHT
    
    normalized_cuboid = cuboid_data.copy()
    
    # print(normalized_cuboid[2])
    normalized_cuboid[2] = (np.float32(normalized_cuboid[2]) - x_min_area_border) / PCD_AREA_WIDTH
    # print(f"norm: {normalized_cuboid[2]}")
    normalized_cuboid[3] = (np.float32(normalized_cuboid[3]) - y_min_area_border) / PCD_AREA_HEIGHT
    
    normalized_cuboid[4] = np.float32(normalized_cuboid[4]) / PCD_AREA_WIDTH
    normalized_cuboid[5] = np.float32(normalized_cuboid[5]) / PCD_AREA_HEIGHT
    
    return normalized_cuboid


def get_labels(cuboid_data, yaw_bias):
    print(yaw_bias)
    label_data = np.zeros((cuboid_data.shape[0], 9))

    label_data[:, 0] = get_label_indexes(cuboid_data[:, 0])

    xywhr = np.zeros((cuboid_data.shape[0], 5))
    xywhr[:, :4] = cuboid_data[:, 2:6]
    xywhr[:, 4] = cuboid_data[:, 1] 

    cuboid_points = xywhr_to_4xy(xywhr)
    label_data[:, 1:] = cuboid_points

    return label_data

In [239]:
# Определяем края для обучения
# x_min = np.min(cuboids_array[:, 2])
# x_max = np.max(cuboids_array[:, 2])
# x_max_dim = np.max(cuboids_array[:, 4])

# y_min = np.min(cuboids_array[:, 3])
# y_max = np.max(cuboids_array[:, 3])
# y_max_dim = np.max(cuboids_array[:, 5])

x_min_border = -WIDTH_MAGIC
x_max_border = WIDTH_MAGIC

y_min_border = -HEIGHT_MAGIC
y_max_border = HEIGHT_MAGIC

print(f"Зона для обучения: [({x_min_border}, {y_min_border}) : ({x_max_border}, {y_max_border})]")

# y (cols)
# |
# |
# 0 --- x  (rows)

row_num = int((x_max_border - x_min_border - 1) // PCD_AREA_WIDTH + 1)
col_num = int((y_max_border - y_min_border - 1) // PCD_AREA_HEIGHT + 1)
print(f"{row_num}x{col_num} зон для обучения")

# Для нормализации z
z_min = np.min(points_array[:, 2])
z_max = np.max(points_array[:, 2])

Зона для обучения: [(-50, -50) : (50, 50)]
2x2 зон для обучения


In [240]:
# Распределение всех точек по необходимым зонам (row_num x col_num)
def get_point_areas(points_array, row_num, col_num, x_min_border, y_min_border):
    point_areas = [[[] for _ in range((col_num))] for _ in range(row_num)]

    for point in points_array:
        
        x_idx = int((point[0] - x_min_border) // PCD_AREA_WIDTH)
        y_idx = int((point[1] - y_min_border) // PCD_AREA_HEIGHT)
        if x_idx >= 0 and y_idx >= 0 and x_idx < row_num and y_idx < col_num:
            point_areas[x_idx][y_idx].append(point)
    
    return point_areas

def get_cuboid_areas(cuboids_array, row_num, col_num, x_min_border, y_min_border):
    cuboid_areas = [[[] for _ in range((col_num))] for _ in range(row_num)]

    for cuboid in cuboids_array:
        x_idx = int((cuboid[2] - x_min_border) // PCD_AREA_WIDTH)
        y_idx = int((cuboid[3] - y_min_border) // PCD_AREA_HEIGHT)
        if x_idx >= 0 and y_idx >= 0 and x_idx < row_num and y_idx < col_num:
            normalize_cuboid = get_normalized_cuboid(cuboid, x_idx, y_idx, x_min_border, y_min_border)
            cuboid_areas[x_idx][y_idx].append(normalize_cuboid)
    
    return cuboid_areas

point_areas = get_point_areas(points_array, row_num, col_num, x_min_border, y_min_border)
# print(cuboids_array)
cuboid_areas = get_cuboid_areas(cuboids_array, row_num, col_num, x_min_border, y_min_border)
# print(cuboid_areas)
# print(len(point_areas))
# print(np.max(np.asarray(point_areas[0][0])[:, 0]))


In [241]:
# Create (x, y): (counts, maxZ) dict
def get_CoordToCountValInt_dict(points):
    coord_to_countval = dict()
    for point in points:
        if coord_to_countval.get((point[0], point[1])) == None:
            coord_to_countval[((point[0], point[1]))] = [1, point[2], point[3]]
        else:
            coord_to_countval[((point[0], point[1]))][0] += 1
    return coord_to_countval


def pcd_to_img_map(points_array, x_idx, y_idx, x_min_border, y_min_border, z_min, z_max):
    x_min_area_border = x_min_border + x_idx * PCD_AREA_WIDTH
    y_min_area_border = y_min_border + y_idx * PCD_AREA_HEIGHT

    # Нормализуем x,y [0: 1]
    points_array[:, 0] = (points_array[:, 0] - x_min_area_border) / PCD_AREA_WIDTH
    points_array[:, 1] = (points_array[:, 1] - y_min_area_border) / PCD_AREA_HEIGHT

    # Приводим x,y к [0: BEV_H/BEV_W]
    points_array[:, 0] = np.int32(points_array[:, 0] * BEV_WIDTH)
    points_array[:, 1] = np.int32(points_array[:, 1] * BEV_HEIGHT)

    # Приводим z к [0, 1]
    points_array[:, 2] = (points_array[:, 2] - z_min) / (z_max - z_min)

    # Сортируем (увел х, увел y, умен z)
    ix = np.lexsort((-points_array[:, 2], points_array[:, 1], points_array[:, 0]))
    points_array = points_array[ix]

    height_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))
    density_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))
    intensity_map = np.zeros((BEV_HEIGHT, BEV_WIDTH))

    points_array[:, 0] = np.minimum(np.maximum(points_array[:, 0], 0), BEV_HEIGHT - 1)
    points_array[:, 1] = np.minimum(np.maximum(points_array[:, 1], 0), BEV_WIDTH - 1)

    coord_to_countval = get_CoordToCountValInt_dict(points_array)
    # points_array = points_array[ix]

    for ((x, y), (c, z, i)) in coord_to_countval.items():
        density_map[int(x)][int(y)] = min(1.0, np.log(c + 1) / np.log(64))
        height_map[int(x)][int(y)] = z
        intensity_map[int(x)][int(y)] = i

    img_map = np.zeros([BEV_HEIGHT, BEV_WIDTH, 3])
    img_map[:,:,0] = density_map
    img_map[:,:,1] = height_map
    # img_map[:,:,2] = height_map
    # img_map[:,:,2] = np.full([BEV_HEIGHT, BEV_WIDTH ], 1)
    img_map[:,:,2] = intensity_map
    
    return img_map

# temp = np.asarray(point_areas[0][0])
# img = pcd_to_img_map(temp, 0, 0, x_min_border, y_min_border, z_min, z_max)

# bevImage = img * 255

# cv2.imshow("BEV", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# cv2.imwrite("test.png", bevImage.astype(np.uint8))
# img = Image.fromarray(bevImage.astype(np.uint8))
# img

In [242]:
for x in range(0, row_num):
    for y in range(0, col_num):
        cur_area_point_arr = np.asarray(point_areas[x][y])
        cur_area_cuboid_arr = np.asarray(cuboid_areas[x][y])

        if len(cur_area_cuboid_arr) * len(cur_area_point_arr) == 0:
            continue
        
        img = pcd_to_img_map(cur_area_point_arr, x, y, x_min_border, y_min_border, z_min, z_max)
        bevImage = img * 255
        
        # print(cur_area_cuboid_arr[:, 2:] * 640)
        
        # print(cur_area_cuboid_arr[:, 1])
        print(cur_area_cuboid_arr[:, 1])
        area_labels = get_labels(cur_area_cuboid_arr, yaw_bias)
        area_labels = area_labels[:, 1:] * 640
        # print(area_labels)
        
        for cuboid in area_labels:
            # cnt = np.array([cuboid[0:2][::-1], cuboid[2:4][::-1], cuboid[4:6][::-1], cuboid[6:8][::-1]], dtype=np.int32)
            cnt = np.array([cuboid[0:2], cuboid[4:6], cuboid[6:8], cuboid[2:4]], dtype=np.int32)
            # print(points)
            cv2.drawContours(img, [cnt], -1, (0, 255, 0), 3)
        cv2.imshow("Contour by 4 points", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
            
        cv2.imwrite(f"test{x}{y}.png", bevImage.astype(np.uint8))


[2.387817238242906 2.3529106532030197 2.3354573606830766 2.370363945722963
 -0.7902868306376103 -0.8251934156774965 2.368759115472126
 -0.7728335381176672 -0.8340980855201181 -0.7728335381176672
 -0.7902868306376103 -0.7972175330910423 -0.7902868306376103
 -0.7972175330910423 2.368759115472126 -0.7972175330910423]
-0.7903721575593857
[2.3529106532030197 2.3354573606830766]
-0.7903721575593857
[2.368759115472126 -0.7876845447168703 -1.4407527169035506
 -1.4407527169035506 -1.4232994243836075 -1.2662197917041178]
-0.7903721575593857
[-0.8400444222766996 -0.7876845447168703 2.370363945722963
 2.3529106532030197 -0.7527779596769841 -0.7702312521969272
 -0.7702312521969272 -0.7702312521969272 -0.7004180821171531
 -0.7876845447168703 -0.7527779596769841 -0.7702312521969272
 2.379281705538637 -0.7797642405710992 -0.7797642405710992
 -0.7623109480511561 -0.7623109480511561 -0.7797642405710992
 -0.7797642405710992 -0.7623109480511561 2.396734998058582
 -0.7427492978012342]
-0.7903721575593857


У 001 и 014.004 и 030 разный биас (0 и pi / 4 и 5pi / 12)

yaw относительно камеры!!


По каждой папке отдельно по всем кубоидам. Предусмотреть разворот и удаление.


In [ ]:
import json
from datetime import datetime
import math


def poses_to_rads(poses_data):
    rads = []
    for pose_data in poses_data:
        yaw = get_z_rotation(pose_data['heading'])
        rads.append(yaw)
    
    return np.asarray(rads)

    

def get_z_rotation(heading):
    w = heading['w']
    x = heading['x']
    y = heading['y']
    z = heading['z']
    vx = 1 - 2 * (y**2 + z**2)
    vy = 2 * (x * y + w * z)
    angle_rad = math.atan2(vy, vx)
    # angle_deg = math.degrees(angle_rad)
    
    return angle_rad


my_test_vals = []

filenames = ['001', '002', '004', '008', '020', '030', '040', '041']

for filename in filenames:
    with open(f"./training/Pandaset/{filename}/lidar/poses.json", 'r') as file:
        poses_data = json.load(file)
        my_test_vals.append(poses_data[0])

yaw_all = [get_z_rotation(x['heading']) for x in my_test_vals]        
# print(my_test_vals)
# z1 = get_z_rotation(my_test_vals[0]['heading'])
# z2 = get_z_rotation(my_test_vals[1]['heading'])
# z3 = get_z_rotation(my_test_vals[2]['heading'])

print(yaw_all)

# print(z1 + np.pi / 4)
# print(f'{abs(z1 - z2)}::::::{np.pi / 4}')
# print(f'{abs(z1 - z3)}::::::{5 * np.pi / 12}')

# print(poses_to_rads(poses_data))

[-0.7944430034154631, -0.7903721575593857, -0.806450082947037, -0.7964144267351548, 0.6322791384360434, -1.3813610438660806, -2.998616796386284, -2.9816808561253767]
